## Creación de `DataFrame` con datos remotos
- Podemos utilizar repositorios en la web para analizar datos remotos, importando sus datos.\
- La forma mas común es utilizando el formato `csv`
- Vamos a utilizar datos de:
  1. **COVID19**: Center for Systems Science and Engineering (CSSE) at Johns Hopkins University
  2. **USGS**: United States Geological Survey
  3. **Placas tectónicas**

**Documentación:**\
[🌍 COVID-19](https://github.com/CSSEGISandData/COVID-19)\
[🌍 USGS](https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php)\
[🌍 Placas tectónicas](https://github.com/fraxen/tectonicplates)

## Importación de librerías

In [ ]:
import requests # realizar solicitudes HTTP.

import io # Decodificar
# Para manipular dirección de carpetas y archivos

import numpy as np # Reemplazar valores vacios por NaN

import pandas as pd # análisis y manipulación de datos.
import folium # visualizar datos geoespaciales


### Request
Podemos usar `request` para solicitar y extraer datos de alguna dirección web con `HTTP`

Por ejemplo de la pagina [https://phet-dev.colorado.edu](https://phet-dev.colorado.edu/html/build-an-atom/0.0.0-3/simple-text-only-test-page.html)

In [ ]:
web_simple = requests.get('https://phet-dev.colorado.edu/html/build-an-atom/0.0.0-3/simple-text-only-test-page.html')
print(web_simple.text)

### Importar un `csv` desde el USGS
Ahora podemos intentar importar en un dataframe el contenido del siguiente link:\
[employees.csv](https://gist.githubusercontent.com/bobbyhadz/9061dd50a9c0d9628592b156326251ff/raw/381229ffc3a72c04066397c948cf386e10c98bee/employees.csv)

In [ ]:
basic_csv = "https://gist.githubusercontent.com/bobbyhadz/9061dd50a9c0d9628592b156326251ff/raw/381229ffc3a72c04066397c948cf386e10c98bee/employees.csv"

urlData = requests.get(basic_csv).content
print(urlData)

data = pd.read_csv(
    io.StringIO(urlData.decode('utf-8')), # Decodificar el utf-8
    sep=',',
    encoding='utf-8',
)
display(data)

 Verificamos el tipo de dato

In [ ]:
type(data)

In [ ]:
display(data)

## Importar los datos de USGS
Todos los sismos en los últimos 30 días

In [ ]:
usgs_url = 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.csv'

usgs_all_data = requests.get(usgs_url).content

usgs_all_df = pd.read_csv(
    io.StringIO(usgs_all_data.decode('utf-8')),
    sep=',',
    encoding='utf-8'
)

display(usgs_all_df)

In [ ]:
print(usgs_all_df.empty)
print('*******************')
print(usgs_all_df.shape)
print('*******************')
print(usgs_all_df.describe())

In [ ]:
usgs_all_df.to_csv('all_earthquakes.csv', na_rep='NaN')

# Ordenar por fecha
sorted_date = usgs_all_df.iloc[:10]
# display(sorted_date)

# Ordenar por sismos
sorted_all_mg = usgs_all_df.sort_values(by='mag', ascending=False)
display(sorted_all_mg)

### Representar datos geoespaciales en un mapa

[Documentacio de Folium](https://python-visualization.github.io/folium/latest/)

In [ ]:
# Crear un mapa
mapa = folium.Map(location=(9.275101, -79.871888), zoom_start=1)
folium.Marker(location=(9.275101, -79.871888)).add_to(mapa)
folium.Circle(
    location=(9.1, -79.871888),
    radius=10000, 
    color='red', 
    fill_color='blue'
    ).add_to(mapa)

ultimo_sismo_lat = usgs_all_df['latitude'].iloc[0]
ultimo_sismo_lon = usgs_all_df['longitude'].iloc[0]
ultimo_sismo_mag = usgs_all_df['mag'].iloc[0]
folium.Circle(
    location=(ultimo_sismo_lat,ultimo_sismo_lon),
    radius=ultimo_sismo_mag*10000, 
    color='red', 
    fill_color='blue'
    ).add_to(mapa)

mapa.save('mapa1.html')


In [ ]:
for i in range(len(usgs_all_df)):
    if usgs_all_df['mag'].iloc[i]>6:
        folium.Circle(location=(usgs_all_df['latitude'].iloc[i], usgs_all_df['longitude'].iloc[i]),
                  radius=usgs_all_df['mag'].iloc[i]*30000,
                  stroke=False,
                  weight=0,
                 fill_color='red',
                 fill_opacity=0.3,
                  tooltip="{mag:.2f} {magT}\n{lugar}".format(mag=usgs_all_df['mag'][i], magT=usgs_all_df['magType']
[i],lugar=usgs_all_df['place'][i]),
                  popup="{} pixels".format(usgs_all_df['mag'][i]),
                 ).add_to(mapa)
    else:
        folium.Circle(location=(usgs_all_df['latitude'].iloc[i], usgs_all_df['longitude'].iloc[i]),
                  radius=usgs_all_df['mag'].iloc[i]*30000,
                  stroke=False,
                  weight=0,
                 fill_color='yellow',
                 fill_opacity=0.3,
                  tooltip="{mag:.2f} {magT}\n{lugar}".format(mag=usgs_all_df['mag'][i], magT=usgs_all_df['magType']
[i],lugar=usgs_all_df['place'][i]),
                  popup="{} pixels".format(usgs_all_df['mag'][i]),
                 ).add_to(mapa)

    folium.Circle(location=(usgs_all_df['latitude'].iloc[i], usgs_all_df['longitude'].iloc[i]),
                  radius=2000, # En metros 
                  stroke=False,
                  fill_opacity = 1, 
                  fill_color='black').add_to(mapa)

mapa.save('mapa2.html')
mapa

In [ ]:
url_placas = 'https://raw.githubusercontent.com/fraxen/tectonicplates/master/GeoJSON/PB2002_boundaries.json'

folium.GeoJson(url_placas, name='geojson').add_to(mapa)
# folium.TileLayer('Mapbox Control Room').add_to(mapa)

mapa